[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/02-python-for-data-science/pyds-file-io.ipynb)

# File Input & Output

*AIBits Academy · Machine Learning End To End · Python For Data Science*

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

Data lives in files. This page covers reading and writing the formats you'll meet daily — plain text, CSV, Excel, JSON, and pickled binary — plus a technique for files too large to fit in memory.

## Text Files & the `with` Statement

Python's built-in `open()` handles text files. Always use it inside a `with` block — the file is closed automatically, even if an error occurs. Modes: `'r'` read, `'w'` write (overwrites), `'a'` append.

In [ ]:
# write, then read back
with open('notes.txt', 'w') as f:
    f.write("Line 1: intro\nLine 2: body\nLine 3: end\n")

with open('notes.txt', 'r') as f:
    content = f.read()
print(repr(content))

# iterate line by line (memory-friendly)
with open('notes.txt', 'r') as f:
    for i, line in enumerate(f, 1):
        print(f"{i}: {line.strip()}")

## CSV Files

CSV is the universal tabular format. The built-in `csv` module reads and writes row by row; `newline=''` avoids blank lines on Windows.

In [ ]:
import csv

rows = [['City', 'Sales'], ['Mumbai', 340], ['Surat', 120], ['Pune', 90]]
with open('sales.csv', 'w', newline='') as f:
    csv.writer(f).writerows(rows)

with open('sales.csv', 'r') as f:
    for row in csv.reader(f):
        print(row)

In practice you'll almost always use **pandas** instead — one line each way, and you get a DataFrame.

In [ ]:
import pandas as pd

df = pd.DataFrame({'City': ['Mumbai', 'Surat', 'Pune'], 'Sales': [340, 120, 90]})
df.to_csv('sales_pd.csv', index=False)     # write
print(pd.read_csv('sales_pd.csv'))         # read

## Excel Files

Excel is ubiquitous in business. Pandas reads and writes `.xlsx` with `read_excel`/`to_excel` (needs the `openpyxl` package). You can also target specific sheets with `sheet_name`.

In [ ]:
import pandas as pd

df.to_excel('sales.xlsx', index=False)          # write (pip install openpyxl)
print(pd.read_excel('sales.xlsx'))              # read

## JSON Files

JSON is the language of web APIs and config files. The `json` module maps directly to Python dicts and lists; `indent=` makes the output human-readable.

In [ ]:
import json

data = {'city': 'Mumbai', 'pincodes': [400001, 400002], 'active': True}
with open('data.json', 'w') as f:
    json.dump(data, f, indent=2)                # write

with open('data.json', 'r') as f:
    loaded = json.load(f)                        # read
print(loaded)
print("nested access:", loaded['pincodes'][0])

For a *list of records*, `pd.json_normalize` flattens JSON straight into a DataFrame.

In [ ]:
import pandas as pd

records = [{'name': 'Anaya', 'city': 'Mumbai'},
           {'name': 'Rohan', 'city': 'Surat'}]
print(pd.json_normalize(records))

## Binary Files & Pickle

To save a Python object exactly as-is — a trained model, a nested structure — use `pickle`. It writes **binary**, so open with `'wb'`/`'rb'`.

> **⚠ Security note**
>
> Only unpickle files you trust — a malicious pickle can execute arbitrary code on load.

In [ ]:
import pickle

obj = {'model': 'demo', 'weights': [0.1, 0.2, 0.3]}
with open('model.pkl', 'wb') as f:
    pickle.dump(obj, f)                          # write binary

with open('model.pkl', 'rb') as f:
    restored = pickle.load(f)                    # read binary
print(restored)

## Handling Large Files with Chunking

When a file is too big to fit in memory, read it in **chunks**. `pd.read_csv(..., chunksize=n)` yields DataFrames of `n` rows at a time, so you process the file piece by piece.

In [ ]:
import pandas as pd

# a large file written once...
pd.DataFrame({'x': range(1000), 'y': range(1000, 2000)}).to_csv('big.csv', index=False)

# ...processed 250 rows at a time
total = 0
for chunk in pd.read_csv('big.csv', chunksize=250):
    total += len(chunk)          # do real work per chunk here
print("rows processed in chunks:", total)

> **✅ What You Can Now Do**
>
> You can read and write text, CSV, Excel, JSON, and pickled binary files, flatten JSON records into DataFrames, and stream files too large for memory with chunking. Next: a focused data-cleaning & preparation workflow.

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Write and read a text log

Write the three strings in `events` to `events.log`, one per line, using `with open(...)`. Read the file back and store the number of lines in `n_lines`.

In [ ]:
events = ["job started", "job progress 50%", "job finished"]
n_lines = None   # TODO


In [ ]:
try:
    check("three lines", n_lines == 3)
    check("content saved", open("events.log").read().splitlines() == events)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
events = ["job started", "job progress 50%", "job finished"]
with open("events.log", "w") as f:
    for e in events:
        f.write(e + "\n")
with open("events.log") as f:
    n_lines = len(f.readlines())

```

</details>

### Exercise 2 · Medium · JSON round trip

Save the dict `config` to `config.json` (indented) and load it back into `loaded`. Then flatten a list of nested records into a DataFrame `flat` with `pd.json_normalize`.

In [ ]:
import json, pandas as pd
config = {"model": "rf", "params": {"n_estimators": 200, "depth": 6}}
records = [{"id": 1, "geo": {"city": "Pune"}}, {"id": 2, "geo": {"city": "Surat"}}]
loaded = flat = None   # TODO


In [ ]:
try:
    check("round trip is lossless", loaded == config)
    check("flattened column name", "geo.city" in flat.columns)
    check("values", flat["geo.city"].tolist() == ["Pune", "Surat"])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import json, pandas as pd
config = {"model": "rf", "params": {"n_estimators": 200, "depth": 6}}
records = [{"id": 1, "geo": {"city": "Pune"}}, {"id": 2, "geo": {"city": "Surat"}}]
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)
with open("config.json") as f:
    loaded = json.load(f)
flat = pd.json_normalize(records)

```

</details>

### Exercise 3 · Stretch · Process a big file in chunks

`orders_big.csv` is created for you with 10,000 rows. Without loading it all at once (`chunksize=2500`), compute the **total of the `amount` column** in `grand_total` and the number of chunks in `n_chunks`.

In [ ]:
import pandas as pd
pd.DataFrame({"id": range(10000), "amount": [i % 100 for i in range(10000)]}).to_csv("orders_big.csv", index=False)
grand_total = n_chunks = None   # TODO


In [ ]:
try:
    check("grand total", grand_total == 495000)
    check("four chunks", n_chunks == 4)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
pd.DataFrame({"id": range(10000), "amount": [i % 100 for i in range(10000)]}).to_csv("orders_big.csv", index=False)
grand_total, n_chunks = 0, 0
for chunk in pd.read_csv("orders_big.csv", chunksize=2500):
    grand_total += chunk["amount"].sum()
    n_chunks += 1

```

</details>

---
*Back to the course: **Machine Learning End To End → File Input & Output**.*